# QLoRA Fine-Tuning: ML Interview Assistant

Fine-tunes Mistral-7B on `win-wang/Machine_Learning_QA_Collection` (~8.6k ML Q&A pairs) using QLoRA.  
**Runtime required:** GPU — T4 (free tier, ~45 min) or A100 (Colab Pro, ~12 min).  

**Setup:** Runtime → Change runtime type → GPU

In [ ]:
# 1. Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 2. Install dependencies
!pip install -q transformers peft trl bitsandbytes accelerate datasets scipy pyyaml

In [ ]:
# 3. Authenticate with HuggingFace (needed for Mistral gated model)
# Get token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# 4. Load + parse win-wang/Machine_Learning_QA_Collection
# Source format: Gemma-style chat turns <start_of_turn>user\n...\n<start_of_turn>model\n...
# This cell converts to instruction/output Alpaca format.

import re, json, os
from datasets import load_dataset, Dataset

DATASET_ID = "win-wang/Machine_Learning_QA_Collection"
MAX_TRAIN = 5000   # cap for a faster demo run; remove to use all ~8600

def parse_turn(text):
    user = re.search(r"<start_of_turn>user\s*(.*?)<end_of_turn>", text, re.DOTALL)
    model = re.search(r"<start_of_turn>model\s*(.*?)(?:<end_of_turn>|$)", text, re.DOTALL)
    if not user or not model:
        return None
    q, a = user.group(1).strip(), model.group(1).strip()
    if len(q) < 20 or len(a) < 30:
        return None
    return {"instruction": q, "input": "", "output": a}

raw = load_dataset(DATASET_ID)

def parse_split(split, limit=None):
    examples = []
    for row in split:
        ex = parse_turn(row["text"])
        if ex:
            examples.append(ex)
        if limit and len(examples) >= limit:
            break
    return examples

train_examples = parse_split(raw["train"], MAX_TRAIN)
val_examples = parse_split(raw["validation"])

os.makedirs("data", exist_ok=True)
for path, data in [("data/train.jsonl", train_examples), ("data/val.jsonl", val_examples)]:
    with open(path, "w") as f:
        for ex in data:
            f.write(json.dumps(ex) + "\n")

print(f"Train: {len(train_examples)} | Val: {len(val_examples)}")
print("\nSample Q:", train_examples[0]["instruction"][:120])
print("Sample A:", train_examples[0]["output"][:200])

In [ ]:
# 5. Config
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR = "./checkpoints"
MAX_SEQ_LEN = 512

LORA_R = 16        # experiment: 8, 16, 64
LORA_ALPHA = 32

USE_BF16 = torch.cuda.is_bf16_supported()   # A100 → True; T4 → False
print(f"Using {'bf16' if USE_BF16 else 'fp16'}")

In [ ]:
# 6. Load tokenizer + 4-bit quantized base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"Loaded {model.num_parameters():,} parameters")

In [ ]:
# 7. Apply LoRA adapters
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~4M trainable / ~3.8B total (0.1%) for r=16

In [ ]:
# 8. Format dataset for SFTTrainer
SYSTEM = (
    "You are an expert ML engineer. Answer the following question clearly "
    "and concisely, as you would in a technical interview."
)

def format_prompt(example):
    return {
        "text": (
            f"<s>[INST] <<SYS>>\n{SYSTEM}\n<</SYS>>\n\n"
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']} [/INST]</s>"
        )
    }

train_data = Dataset.from_list(train_examples).map(format_prompt)
val_data = Dataset.from_list(val_examples).map(format_prompt)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print(train_data[0]["text"][:500])

In [ ]:
# 9. Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.001,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_32bit",
)

In [ ]:
# 10. Train
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
    peft_config=lora_config,
)

print("Starting training...")
trainer.train()
print("Done.")

In [ ]:
# 11. Save adapter
adapter_path = f"{OUTPUT_DIR}/final_r{LORA_R}"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved: {adapter_path}")

In [ ]:
# 12. Qualitative eval — fine-tuned answers
from peft import PeftModel

EVAL_QUESTIONS = [
    "What is LoRA and why is it more parameter-efficient than full fine-tuning?",
    "What is the difference between RAG and fine-tuning?",
    "Explain the vanishing gradient problem and how to fix it.",
    "How do you handle class imbalance in a dataset?",
    "What is the attention mechanism in transformers?",
]

def generate(mdl, tok, question, max_new=300):
    prompt = (
        f"<s>[INST] <<SYS>>\n{SYSTEM}\n<</SYS>>\n\n"
        f"### Instruction:\n{question}\n\n### Response:\n"
    )
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs, max_new_tokens=max_new,
            temperature=0.7, top_p=0.9, do_sample=True,
            eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

ft_model = PeftModel.from_pretrained(model, adapter_path)
ft_model.eval()

for q in EVAL_QUESTIONS:
    print(f"\n{'='*60}\nQ: {q}")
    print(generate(ft_model, tokenizer, q))

In [ ]:
# 13. LoRA rank experiment — rerun cells 7-12 with different LORA_R
# Record results:
#
# | r  | Trainable params | Val loss | Notes                     |
# |----|-----------------|----------|---------------------------|
# | 8  | ?               | ?        |                           |
# | 16 | ?               | ?        |                           |
# | 64 | ?               | ?        |                           |
print("Fill in table after running r=8, r=16, r=64")

In [ ]:
# 14. Save to Google Drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(adapter_path, f'/content/drive/MyDrive/lora-adapters/final_r{LORA_R}')
# print('Saved to Google Drive')